# E8: ngưỡng tối ưu theo chi phí overbooking

**Câu hỏi:** nếu khách sạn dùng xác suất hủy để quyết định overbooking, ngưỡng nào làm tổng chi phí nhỏ nhất?

**Mô hình chi phí (toàn bộ là GIẢ ĐỊNH của nhóm, không phải số liệu kinh doanh thật)**
- Dự đoán hủy, nhưng khách vẫn đến (FP): chi phí `c_walk` (chuyển khách, bồi thường).
- Dự đoán không hủy, nhưng khách hủy (FN): chi phí `c_empty` (phòng trống, mất doanh thu).
- Đặt `c_empty = 1`, thử nhiều tỷ lệ `r = c_walk / c_empty`.
- Ngưỡng lý thuyết khi xác suất đã hiệu chỉnh: `t* = r / (r + 1)`.

**Quy tắc:** chỉ dùng `val` để chọn ngưỡng. Không mở `test` trong notebook này.

In [ ]:
import sys; sys.path.insert(0, "..")

from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

from src.data import load_xy
from src.evaluate import load_proba, evaluate, log_result
from src.cost import (cost_curve, best_threshold, cost_table, cost_table_by_group,
                      theoretical_threshold)

AUTHOR = "Tran_Hoang_Tung"          # giữ giống notebook 04 của bạn
MODEL = "mlp_balanced"                   # đổi thành mô hình tốt nhất khi có (vd "mlp", "xgb")
RATIOS = [0.5, 1, 2, 3, 5, 10]     # r = c_walk / c_empty
R_MAIN = 2                         # mức r dùng cho phân tích chi tiết, bạn chọn và ghi lý do trong báo cáo
FIG_DIR = Path("../reports/figures"); FIG_DIR.mkdir(parents=True, exist_ok=True)

## Đọc dữ liệu val và xác suất của mô hình

In [ ]:
X_val, y_val = load_xy("val")
proba = load_proba(MODEL, split="val")
assert len(proba) == len(y_val), "Số dòng proba không khớp val.csv: kiểm tra MODEL / thứ tự dòng"
print(f"{MODEL}: {len(y_val)} đơn, tỷ lệ hủy {y_val.mean():.1%}")

## 1. Ngưỡng tối ưu và chi phí theo từng mức r

- `gap_vs_theory` lệch xa 0 nghĩa là xác suất chưa được hiệu chỉnh tốt (nối với E7 và hiện tượng data drift).
- `saving_vs_never`: % chi phí giảm so với không bao giờ overbooking.

In [ ]:
table = cost_table(y_val, proba, ratios=RATIOS)
table.round(3)

## 2. Đường cong chi phí

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex=True)
for ax, r in zip(axes.ravel(), RATIOS):
    curve = cost_curve(y_val, proba, c_walk=r)
    b = best_threshold(curve)
    ax.plot(curve["threshold"], curve["cost"], color="tab:blue")
    ax.axvline(b["threshold"], color="tab:green", ls="--", label=f"tối ưu t={b['threshold']:.2f}")
    ax.axvline(theoretical_threshold(r), color="tab:red", ls=":", label=f"lý thuyết t*={theoretical_threshold(r):.2f}")
    never = table.loc[table.r == r, "cost_never"].iloc[0]
    ax.axhline(never, color="gray", lw=1, label="không overbooking")
    ax.set_ylim(b["cost"] * 0.9, never * 1.3)   # zoom vào vùng quanh điểm tối ưu
    ax.set_title(f"r = c_walk/c_empty = {r}"); ax.legend(fontsize=8)
for ax in axes[1]: ax.set_xlabel("ngưỡng")
for ax in axes[:, 0]: ax.set_ylabel("tổng chi phí (đơn vị c_empty)")
fig.suptitle(f"E8: chi phí theo ngưỡng ({MODEL}, tập val)")
fig.tight_layout(); fig.savefig(FIG_DIR / f"e8_cost_curves_{MODEL}.png", dpi=150); plt.show()

## 3. Ngưỡng tối ưu so với ngưỡng lý thuyết

Nếu hai đường trùng nhau, xác suất đáng tin để tính chi phí. Lệch nhiều thì cần hiệu chỉnh (E7).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(table["r"], table["t_theory"], "o:", color="tab:red", label="lý thuyết t*")
ax.plot(table["r"], table["t_best"], "o-", color="tab:green", label="tối ưu trên val")
ax.set_xscale("log"); ax.set_xlabel("r = c_walk / c_empty"); ax.set_ylabel("ngưỡng"); ax.legend()
ax.set_title(f"Ngưỡng tối ưu và lý thuyết ({MODEL})")
fig.tight_layout(); fig.savefig(FIG_DIR / f"e8_threshold_vs_theory_{MODEL}.png", dpi=150); plt.show()

## 4. Tách theo khách sạn (tại r = R_MAIN)

In [ ]:
cost_table_by_group(y_val, proba, X_val["hotel"], ratio=R_MAIN).round(3)

## 5. Ghi kết quả

Ghi một dòng cho mỗi mức r, tại ngưỡng tối ưu, vào `reports/results/<AUTHOR>.csv`. Chạy lại thì dòng cũ được thay.

In [ ]:
for _, row in table.iterrows():
    m = evaluate(y_val, proba, threshold=row["t_best"])
    m.update(cost=row["cost_best"], cost_never=row["cost_never"], r=row["r"])
    log_result(AUTHOR, "E8", MODEL, "val", m, note=f"r={row['r']}, t_best={row['t_best']}")
print("Đã ghi", len(table), "dòng E8")

In [ ]:
res = pd.read_csv("../reports/results/Tran_Hoang_Tung.csv")
e8 = res[res.experiment == "E8"].copy()
e8["saving"] = 1 - e8["cost"] / e8["cost_never"]   # % chi phí giảm so với không overbooking

print("Mức tiết kiệm so với không overbooking:")
display(e8.pivot(index="r", columns="model", values="saving").round(3))
print("Ngưỡng tối ưu trên val:")
display(e8.pivot(index="r", columns="model", values="threshold").round(2))

## Việc tiếp theo
- Đổi `MODEL` sang các mô hình khác (khi người 2 và 3 lưu xong `proba_val_*.npy`) rồi chạy lại để so sánh.
- Dùng xác suất đã hiệu chỉnh của người 3 (E7) và xem `gap_vs_theory` có nhỏ lại không.
- Ở bước đánh giá cuối (cả nhóm cùng chạy, một lần): áp ngưỡng đã chọn trên val vào test bằng `cost_at_threshold`.